In [1]:
import os
import sys

# 현재 작업 디렉토리 기준으로 상위 1단계 폴더를 루트로 설정
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# print("프로젝트 루트로 설정된 경로:", project_root)

# 데이터 확인하기 2025.11.20 152개 컬럼을 사용해 RandomForest 기본 모델 돌리기
import pandas as pd
import numpy  as np

from sklearn.preprocessing import StandardScaler # 데이터 전처리용

import matplotlib.pyplot as plt
import seaborn as sns

# import importlib
# from utils import preprocessing, user_utils

# # 모듈 reload
# importlib.reload(user_utils)

from utils.preprocessing import load_data, split_features_target
from utils.model_utils   import save_model
from utils.user_utils    import get_clf_eval


import warnings

warnings.filterwarnings("ignore", category=UserWarning)


c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 원본 데이터 불러오기
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

In [3]:
remove_cols = pd.read_csv('../doc/remove_cols.xls', header=0).squeeze()
train.drop(columns=remove_cols, axis=1, inplace=True)
test.drop(columns=remove_cols, axis=1, inplace=True)

In [4]:
X_features = train.drop(columns=['ID', 'TARGET'], axis=1) # ID와 TARGET 모두 제거하고 X_train 만들기
y_labels = train['TARGET']
X_test = test.drop(columns=['ID'], axis=1) # test데이터 ID 제거
X_features.columns

Index(['var3', 'var15', 'imp_ent_var16_ult1', 'imp_op_var39_comer_ult1',
       'imp_op_var39_comer_ult3', 'imp_op_var41_comer_ult1',
       'imp_op_var41_comer_ult3', 'imp_op_var41_efect_ult1',
       'imp_op_var41_efect_ult3', 'imp_op_var41_ult1',
       ...
       'saldo_medio_var8_ult3', 'saldo_medio_var12_hace2',
       'saldo_medio_var12_hace3', 'saldo_medio_var12_ult1',
       'saldo_medio_var12_ult3', 'saldo_medio_var13_corto_hace2',
       'saldo_medio_var13_corto_hace3', 'saldo_medio_var13_corto_ult1',
       'saldo_medio_var13_corto_ult3', 'var38'],
      dtype='object', length=149)

In [5]:
# Data 전처리 2. var3 의 최소값 -99999 를 최빈값으로 변경하기
X_features['var3'] = X_features['var3'].replace(-999999, 2)

In [6]:
# 152개 컬럼의 데이터 전처리한 것 저장해 두기, 다음에 다시 테스트할때 대비
# X_features.to_csv('../data/x_train_149cols.csv')
# X_test.to_csv('../data/x_test_149cols.csv')

In [7]:
scaler        = StandardScaler()
X_scaled      = scaler.fit_transform(X_features)
X_test_scaled = scaler.transform(X_test)

In [8]:
# 레이블의 분포 확인
cust_cnt = y_labels.value_counts()
print(cust_cnt) # 1이 불만족 3008명, 만족이 73012

# 불만족고객의 비율
cust_rate = cust_cnt[1] / cust_cnt.sum()
print(f'불만족 고객 비율: {cust_rate:.2f}')

TARGET
0    73012
1     3008
Name: count, dtype: int64
불만족 고객 비율: 0.04


In [9]:
# type(X_scaled)

In [10]:
# 학습/테스트 데이터 분리
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
  X_features, 
  y_labels,
  test_size    = 0.2, 
  random_state = 23, # 세미프로젝트3조
  stratify = y_labels
)


In [12]:
from lightgbm import LGBMClassifier
lgbm_clf = LGBMClassifier(
    random_state      = 0,
    n_estimators      = 100,
    num_leaves        = 15,        # 가지 수
    min_child_samples = 2
)
lgbm_clf.fit(X_train, y_train)     # 학습
save_model(lgbm_clf, 'LigthGBM_basic')
pred       = lgbm_clf.predict(X_val)   # 예측
pred_proba = lgbm_clf.predict_proba(X_val)[:,1] # 예측확률

get_clf_eval(y_test=y_val, pred=pred, pred_proba=pred_proba)

[LightGBM] [Info] Number of positive: 2406, number of negative: 58410
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012290 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 12178
[LightGBM] [Info] Number of data points in the train set: 60816, number of used features: 149
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.039562 -> initscore=-3.189521
[LightGBM] [Info] Start training from score -3.189521
✓ 모델 저장 완료: models\LigthGBM_basic.pkl
  파일 크기: 0.18 MB
folder = c:\big20\git\big20-ML-project2-team3\SantanderCS\results
AUC: 0.8462, 정확도: 0.9591, 정밀도: 0.2222, 재현율: 0.0133, F1: 0.0251
오차행렬:
[[14574    28]
 [  594     8]]


In [14]:
# RandomForest
# Model 학습, 평가
from sklearn.ensemble import RandomForestClassifier 
from sklearn.metrics  import roc_auc_score, accuracy_score, classification_report, confusion_matrix

# 기본값으로 우선 RF 해 보자
rf_clf = RandomForestClassifier(
  random_state = 0,
  n_estimators = 100,
  max_depth    = 8, # RF : 애가 핵심이야 Tree 계열이니까~ 약한 Tree로 만들어야해 그래서 max_depth로 자른거
  n_jobs       = -1 # 병렬처리 여부 
)
rf_clf.fit(X_train, y_train)    # 학습
pred       = rf_clf.predict(X_val)   # 예측
pred_proba = rf_clf.predict_proba(X_val)[:,1] # 예측확률

print(f'acc : {accuracy_score(y_val, pred)} \nroc_auc : {roc_auc_score(y_val, pred_proba)}')

acc : 0.9604051565377533 
roc_auc : 0.8213161761393446
